# ⚽ Capstone Project: The European Soccer Analytics Agency

**Role:** Junior Data Analyst  
**Agency:** EuroGoal Analytics  
**Database:** European Soccer Database (SQLite)

---

## 📩 Welcome to the Team

Congratulations on your new role at **EuroGoal Analytics**. We provide data-driven insights to the biggest clubs, betting syndicates, and sports news networks in Europe.

Your job is to query our massive database of matches, players, and teams from 2008 to 2016 to answer urgent requests from our stakeholders.

### 🗄️ The Database Schema

Before you begin, familiarize yourself with the data architecture. You will need to join these tables frequently.



1.  **`Country`**: A reference list of countries (e.g., Belgium, England).
2.  **`League`**: The specific competitions (e.g., Premier League) linked to a `country_id`.
3.  **`Match`**: The core transactional table. Contains `home_team_api_id`, `away_team_api_id`, scores, and dates.
4.  **`Team`**: Details about the clubs (`team_long_name`) linked by `team_api_id`.
5.  **`Player`**: Biographical data (Name, Height, Birthday) linked by `player_api_id`.
6.  **`Player_Attributes`**: Historical stats (Rating, Potential) linked to `Player` by `player_api_id`.
7.  **`Team_Attributes`**: Historical tactical data (Build Up Play, Defense) linked to `Team`.

---

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("database.sqlite")

## 📥 Section 1: The Press Room Requests
*Incoming queries from journalists and broadcasters who need quick facts for their stories.*

**1. The Coverage Map** 
"We need to update our website's footer. Please list every single **Country** name we have data for, sorted alphabetically."

**Expected Output:**
| id | name |
|---|---|
| 1 | Belgium |
| 1729 | England |
| ... | ... |

In [115]:
pd.read_sql(
  """
    SELECT *
    FROM Country
    ORDER BY name ASC
  """,
  conn
)

,id,name
0,1,Belgium
1,1729,England
2,4769,France
3,7809,Germany
4,10257,Italy
5,13274,Netherlands
6,15722,Poland
7,17642,Portugal
8,19694,Scotland
9,21518,Spain


**2. The Team Roster** 
"We are building a dropdown menu for the search bar. Give me a list of every unique **Team Long Name** in the database."

**Expected Output:**
| team_long_name |
|---|
| Arsenal |
| Aston Villa |
| ... |

In [116]:
pd.read_sql(
  """
    SELECT DISTINCT team_long_name
    FROM Team
    WHERE team_long_name LIKE 'A%'
  """,
  conn
)

,team_long_name
0,Arsenal
1,Aston Villa
2,AJ Auxerre
3,AS Monaco
4,AS Nancy-Lorraine
5,AS Saint-Étienne
6,AC Arles-Avignon
7,AC Ajaccio
8,Angers SCO
9,Atalanta


**3. The Giants Feature** 
"I'm writing a piece on the tallest athletes in football. Find me the Player Name and Height of anyone taller than **200cm**."

**Expected Output:**
| player_name | height |
|---|---|
| Kristof van Hout | 208.28 |
| ... | ... |

In [117]:
pd.read_sql(
  """
    SELECT player_name, height
    FROM Player
    WHERE height > 200
    ORDER BY height DESC
  """,
  conn
)

,player_name,height
0,Kristof van Hout,208.28
1,Bogdan Milic,203.20
2,Costel Pantilimon,203.20
3,Fejsal Mulic,203.20
4,Jurgen Wevers,203.20
5,Kevin Vink,203.20
6,Lacina Traore,203.20
7,Nikola Zigic,203.20
8,Paolo Acerbis,203.20
9,Pietro Marino,203.20


**4. The Heavyweights** 
"Follow up to the previous request: List the top 10 heaviest players in the database. I need their names and weights, ordered descending."

**Expected Output:**
| player_name | weight |
|---|---|
| Kristof van Hout | 243 |
| Tim Wiese | 243 |
| ... | ... |

In [118]:
pd.read_sql(
  """
    SELECT player_name, weight
    FROM Player
    WHERE weight > 200
    ORDER BY weight DESC
    LIMIT 10
  """,
  conn
)

,player_name,weight
0,Kristof van Hout,243
1,Tim Wiese,243
2,Jeroen Verhoeven,227
3,Ishmael Miller,225
4,Cassio,220
5,Christopher Samba,220
6,Lars Unnerstall,220
7,Marcus Hahnemann,220
8,Abdoulaye Faye,218
9,Enoch Showunmi,218


**5. The Right-Footed Club** 
"We are looking for players who strictly prefer their right foot. List all players where `preferred_foot` is 'right'."

**Expected Output:**
| player_name | preferred_foot |
|---|---|
| Aaron Appindangoye | right |
| ... | ... |

In [72]:
pd.read_sql(
  """
    SELECT DISTINCT P.player_name, A.preferred_foot
    FROM Player AS P
    JOIN Player_Attributes AS A
      ON P.player_api_id = A.player_api_id
    WHERE A.preferred_foot = 'right'
  """,
  conn
)

,player_name,preferred_foot
0,Aaron Appindangoye,right
1,Aaron Cresswell,right
2,Aaron Doran,right
3,Aaron Galindo,right
4,Aaron Hughes,right
...,...,...
8831,Zoran Josipovic,right
8832,Zoran Rendulic,right
8833,Zoumana Camara,right
8834,Zurab Khizanishvili,right


**6. The 2015/2016 Archive** 
"Our archive team needs to tag records. Find all matches played specifically in the **'2015/2016'** season."

**Expected Output:**
| match | season | date |
|---|---|---|
| Barcelone vs Real Madrid | 2015/2016 | 2015-08-01 |
| ... | ... | ... |

In [120]:
pd.read_sql(
  """
    SELECT CONCAT(HT.team_long_name,' vs ',AT.team_long_name) AS match, season, DATE(date) AS date
    FROM Match AS M
    JOIN Team AS HT
      ON M.home_team_api_id = HT.team_api_id
    JOIN Team AS AT
      ON M.away_team_api_id = AT.team_api_id
    WHERE season = '2015/2016'
  """,
  conn
)

,match,season,date
0,Sint-Truidense VV vs Club Brugge KV,2015/2016,2015-07-24
1,KV Kortrijk vs Standard de Liège,2015/2016,2015-07-25
2,KRC Genk vs Oud-Heverlee Leuven,2015/2016,2015-07-25
3,KV Oostende vs KV Mechelen,2015/2016,2015-07-25
4,SV Zulte-Waregem vs Sporting Lokeren,2015/2016,2015-07-25
...,...,...,...
3321,FC St. Gallen vs FC Thun,2015/2016,2015-09-22
3322,FC Vaduz vs FC Luzern,2015/2016,2015-09-23
3323,Grasshopper Club Zürich vs FC Sion,2015/2016,2015-09-23
3324,Lugano vs FC Zürich,2015/2016,2015-09-22


**7. The 'M' Club** 
"We are doing a fun alphabet feature. Find all teams whose long name starts with the letter 'M', but only include those that have participated in matches during the 2015/2016 season. Sort the results alphabetically by team name."

**Expected Output:**
| team_long_name | Season|
|---|---|
| Manchester United |2012/2013|
| Manchester City |2014/2015|
| Malaga CF |2010/2011|
| ... | ... |

In [121]:
pd.read_sql(
  """
    SELECT DISTINCT T.team_long_name, M.season
    FROM Team AS T
    JOIN Match AS M
      ON M.home_team_api_id = T.team_api_id
      OR M.away_team_api_id = T.team_api_id
    WHERE T.team_long_name LIKE 'M%' AND M.season = '2015/2016'
    ORDER BY T.team_long_name ASC
  """,
  conn
)

,team_long_name,season
0,Manchester City,2015/2016
1,Manchester United,2015/2016
2,Milan,2015/2016
3,Montpellier Hérault SC,2015/2016
4,Moreirense FC,2015/2016
5,Motherwell,2015/2016
6,Málaga CF,2015/2016


**8. The Timeline** 
"Just to double check our data integrity: List the distinct **seasons** available in the Match table."

**Expected Output:**
| season |
|---|
| 2008/2009 |
| 2009/2010 |
| ... |

In [122]:
pd.read_sql(
  """
    SELECT DISTINCT season
    FROM Match
  """,
  conn
)

,season
0,2008/2009
1,2009/2010
2,2010/2011
3,2011/2012
4,2012/2013
5,2013/2014
6,2014/2015
7,2015/2016


**9. The Goal Fests** 
"I need highlights. Find all matches where the **Home Team** scored at least **5 goals**."

**Expected Output:**
| match | home_team_goal | away_team_goal |
|---|---|---|
| Manchester City vs Manchester United | 6 | 0 |
| ... | ... | ... |

In [123]:
pd.read_sql(
  """
    SELECT CONCAT(HT.team_long_name, ' vs ', AT.team_long_name) AS match, M.home_team_goal, M.away_team_goal
    FROM Match AS M
    JOIN Team AS HT
      ON M.home_team_api_id = HT.team_api_id
    JOIN Team AS AT
      ON M.away_team_api_id = AT.team_api_id
    WHERE M.home_team_goal >= 5
  """,
  conn
)

,match,home_team_goal,away_team_goal
0,KAA Gent vs RAEC Mons,5,0
1,RSC Anderlecht vs Tubize,5,1
2,RAEC Mons vs FCV Dender EH,5,2
3,Royal Excel Mouscron vs Club Brugge KV,5,1
4,SV Zulte-Waregem vs KV Kortrijk,5,1
...,...,...,...
666,BSC Young Boys vs FC Luzern,5,2
667,FC Luzern vs FC Vaduz,5,1
668,BSC Young Boys vs Lugano,7,0
669,BSC Young Boys vs FC Vaduz,5,4


**10. The Underdogs** 
"We are looking for potential comeback stories. Find the records in the `Player_Attributes` table where the `overall_rating` is the absolute lowest. Limit to the bottom 3."

**Expected Output:**
| player| overall_rating |
|---|---|
| Drogba | 42 |
| Salah | 43 |
| ... | ... |

In [124]:
pd.read_sql(
  """
    SELECT 
      DISTINCT P.player_name AS player, 
      A.overall_rating
    FROM Player_Attributes AS A
    JOIN Player AS P
      ON A.Player_api_id = P.Player_api_id
    WHERE A.overall_rating IS NOT NULL
    ORDER BY A.overall_rating ASC
    LIMIT 3
  """,
  conn
)

,player,overall_rating
0,Francesco Della Rocca,33
1,James Vincent,35
2,Nicky Kuiper,35


--- 
## 📈 Section 2: The Betting Syndicate
*High-volume requests from our odds-making partners who need totals, averages, and trends.*

**11. Match Volume** 
"How busy was each season? Count the total number of matches played in each season."

**Expected Output:**
| season | Total_Matches |
|---|---|
| 2008/2009 | 3326 |
| 2009/2010 | 3200 |
| ... | ... |

In [125]:
pd.read_sql(
  """
    SELECT 
      DISTINCT season,
      COUNT(*) OVER(PARTITION BY season) AS Total_Matches
    FROM Match

    -- or GROUP BY --
  """,
  conn
)

,season,Total_Matches
0,2008/2009,3326
1,2009/2010,3230
2,2010/2011,3260
3,2011/2012,3220
4,2012/2013,3260
5,2013/2014,3032
6,2014/2015,3325
7,2015/2016,3326


**12. The Height Advantage** 
"We are calculating physical stats. Calculate the **average height** of all players in our database."

**Expected Output:**
| Average_Height |
|---|
| 181.867... |

In [126]:
pd.read_sql(
  """
    SELECT AVG(height) AS Average_Height
    FROM Player
  """,
  conn
)

,Average_Height
0,181.867445


**13. League Scoring Potential** 
"Which league is the highest scoring? Find the total number of goals (Home + Away) scored for each `league`."

**Expected Output:**
| league_Name | Total_League_Goals |
|---|---|
| Belgium Jupiler League | 4500 |
| England Premier League | 8200 |
| ... | ... |

In [127]:
pd.read_sql(
  """
    SELECT
      L.name AS League_Name,
      SUM(M.home_team_goal + M.away_team_goal) Total_League_Goals
    FROM Match AS M
    JOIN League AS L
      ON M.league_id = L.id
    GROUP BY L.name
  """,
  conn
)

,League_Name,Total_League_Goals
0,Belgium Jupiler League,4841
1,England Premier League,8240
2,France Ligue 1,7427
3,Germany 1. Bundesliga,7103
4,Italy Serie A,7895
5,Netherlands Eredivisie,7542
6,Poland Ekstraklasa,4656
7,Portugal Liga ZON Sagres,5201
8,Scotland Premier League,4804
9,Spain LIGA BBVA,8412


**14. League Size** 
"Identify leagues that have more than 10 distinct teams in the Match table."

**Expected Output:**
| league_Name | Count_of_Teams |
|---|---|
| Belgium Jupiler League | 20 |
| England Premier League | 20 |
| ... | ... |

In [128]:
pd.read_sql(
  """
    WITH All_Teams AS (
      SELECT league_id, home_team_api_id AS team
      FROM Match
      UNION ALL
      SELECT league_id, away_team_api_id
      FROM Match
    )
    SELECT 
      L.name AS League_Name,
      COUNT(DISTINCT A.team) Count_of_Teams
    FROM All_Teams AS A
    JOIN League AS L
      ON A.league_id = L.id
    GROUP BY L.name
    HAVING Count_of_Teams > 10
  """,
  conn
)

,League_Name,Count_of_Teams
0,Belgium Jupiler League,25
1,England Premier League,34
2,France Ligue 1,35
3,Germany 1. Bundesliga,30
4,Italy Serie A,32
5,Netherlands Eredivisie,25
6,Poland Ekstraklasa,24
7,Portugal Liga ZON Sagres,29
8,Scotland Premier League,17
9,Spain LIGA BBVA,33


**15. The Thriller** 
"I want to bet on 'Over 6.5 Goals'. Find the single match with the highest total goal count (sum of home and away) in history."

**Expected Output:**
| match | Total_Goals |
|---|---|
| Barcelone vs Real Madrid | 12 |

In [129]:
pd.read_sql(
  """
    WITH T AS (
      SELECT 
        CONCAT(HT.team_long_name, ' vs ', AT.team_long_name) AS match, 
        (M.home_team_goal+M.away_team_goal) AS Total_Goals,
        DENSE_RANK() OVER(ORDER BY (M.home_team_goal+M.away_team_goal) DESC) AS DR
      FROM Match AS M
      JOIN Team AS HT
        ON M.home_team_api_id = HT.team_api_id
      JOIN Team AS AT
        ON M.away_team_api_id = AT.team_api_id
    )
    SELECT match, Total_Goals
    FROM T 
    WHERE DR = 1
  """,
  conn
)

,match,Total_Goals
0,Motherwell vs Hibernian,12
1,Real Madrid CF vs Rayo Vallecano,12


**16. Player Quality Control** 
"Calculate the average `overall_rating` for each `player_api_id`. (Note: Players have multiple records over time, we need their career average)."

**Expected Output:**
| player_Name | Avg_Rating |
|---|---|
| Salah | 67.5 |
| ... | ... |

In [130]:
pd.read_sql(
  """
    SELECT P.player_name, ROUND(AVG(overall_rating), 1) AS Avg_Rating
    FROM Player_Attributes AS PA
    JOIN Player AS P
      ON P.player_api_id = PA.player_api_id
    GROUP BY P.player_name
  """,
  conn
)

,player_name,Avg_Rating
0,Aaron Appindangoye,63.6
1,Aaron Cresswell,67.0
2,Aaron Doran,67.0
3,Aaron Galindo,69.1
4,Aaron Hughes,73.2
...,...,...
10843,Zsolt Low,67.6
10844,Zurab Khizanishvili,70.8
10845,Zvjezdan Misimovic,80.0
10846,de Oliveira Cleber Monteiro,66.1


**17. The Draw Specialists** 
"Which season had the most draws? Count matches where `home_team_goal` = `away_team_goal`, grouped by season."

**Expected Output:**
| season | Draw_Count |
|---|---|
| 2011/2012 | 800 |
| ... | ... |

In [131]:
pd.read_sql(
  """
    SELECT 
      season,
      COUNT(*) AS Draw_Count
    FROM Match
    WHERE home_team_goal = away_team_goal
    GROUP BY season
  """,
  conn
)

,season,Draw_Count
0,2008/2009,831
1,2009/2010,814
2,2010/2011,839
3,2011/2012,818
4,2012/2013,853
5,2013/2014,736
6,2014/2015,850
7,2015/2016,855


**18. Home Field Advantage** 
"Find the average number of **home goals** scored per match specifically in the `2013/2014` season."

**Expected Output:**
| Avg_Home_Goals |
|---|
| 1.56 |

In [132]:
pd.read_sql(
  """
    SELECT ROUND(AVG(home_team_goal), 2) AS Avg_Home_Goals
    FROM Match
    WHERE season = '2013/2014'
  """,
  conn
)

,Avg_Home_Goals
0,1.58


**19. Multi-League Countries** 
"Do any countries have more than one league listed in the League table? Group by country and count the leagues."

**Expected Output:**
| country_Name | League_Count |
|---|---|
| England | 1 |
| ... | ... |

In [133]:
pd.read_sql(
  """
    SELECT 
      C.name AS country_Name, 
      COUNT(L.id) AS League_Count
    FROM League AS L
    JOIN Country AS C
      ON L.country_id = C.id
    GROUP BY C.name
  """,
  conn
)

,country_Name,League_Count
0,Belgium,1
1,England,1
2,France,1
3,Germany,1
4,Italy,1
5,Netherlands,1
6,Poland,1
7,Portugal,1
8,Scotland,1
9,Spain,1


**20. The Worst Visitors** 
"Find the `team_Name` of the team that has lost the most **Away Matches** in history."

**Expected Output:**
| team_Name | Losses |
|---|---|
| Liverbool | 87 |
| ... | ... |

In [134]:
pd.read_sql(
  """
    SELECT 
      T.team_long_name AS team_Name,
      COUNT(*) AS Losses
    FROM Match AS M
    JOIN Team AS T
      ON M.away_team_api_id = T.team_api_id
    WHERE home_team_goal > away_team_goal
    GROUP BY T.team_long_name
  """,
  conn
)

,team_Name,Losses
0,1. FC Kaiserslautern,18
1,1. FC Köln,51
2,1. FC Nürnberg,44
3,1. FSV Mainz 05,50
4,AC Ajaccio,30
...,...,...
291,Xerez Club Deportivo,11
292,Zagłębie Lubin,42
293,Zawisza Bydgoszcz,15
294,Évian Thonon Gaillard FC,41


--- 
## 🕵️‍♂️ Section 3: The Scouting Network
*Complex requests requiring you to connect (Join) different data sources.*

**21. The Match Manifest** 
"IDs are useless to us. Retrieve the Match ID, Date, Home Team Name, and Away Team Name for the first 10 matches in the database."

**Expected Output:**
| match_api_id | date | Home_Team | Away_Team |
|---|---|---|---|
| 489042 | 2008-08-17 | Genk | Beerschot AC |
| ... | ... | ... | ... |

In [135]:
pd.read_sql(
  """
    SELECT 
      match_api_id AS match_api_id,
      DATE(date) AS Date,
      HT.team_long_name AS Home_Team,
      AT.team_long_name AS Away_Team
    FROM Match AS M
    JOIN Team AS HT
      ON M.home_team_api_id = HT.team_api_id
    JOIN Team AS AT
      ON M.away_team_api_id = AT.team_api_id
  """,
  conn
)

,match_api_id,Date,Home_Team,Away_Team
0,492473,2008-08-17,KRC Genk,Beerschot AC
1,492474,2008-08-16,SV Zulte-Waregem,Sporting Lokeren
2,492475,2008-08-16,KSV Cercle Brugge,RSC Anderlecht
3,492476,2008-08-17,KAA Gent,RAEC Mons
4,492477,2008-08-16,FCV Dender EH,Standard de Liège
...,...,...,...,...
25974,1992091,2015-09-22,FC St. Gallen,FC Thun
25975,1992092,2015-09-23,FC Vaduz,FC Luzern
25976,1992093,2015-09-23,Grasshopper Club Zürich,FC Sion
25977,1992094,2015-09-22,Lugano,FC Zürich


**22. The Big Three** 
"Filter for matches played in **England**, **Spain**, or **Germany**. You will need to join `Country` and `League` to find them."

**Expected Output:**
| match | Country_Name | League_Name |
|---|---|---|
| Liverbool vs Man City | England | Premier League |
| ... | ... | ... |

In [136]:
pd.read_sql(
  """
    SELECT 
      CONCAT(HT.team_long_name, ' vs ', AT.team_long_name) AS match,
      C.name AS Country_Name,
      L.name AS League_Name
    FROM Match AS M
    JOIN Country AS C
      ON M.country_id = C.id
    JOIN League AS L
      ON M.league_id = L.id
    JOIN Team AS HT
      ON M.home_team_api_id = HT.team_api_id
    JOIN Team AS AT
      ON M.away_team_api_id = AT.team_api_id
    WHERE C.name IN ('England','Spain','Germany')
  """,
  conn
)

,match,Country_Name,League_Name
0,Manchester United vs Newcastle United,England,England Premier League
1,Arsenal vs West Bromwich Albion,England,England Premier League
2,Sunderland vs Liverpool,England,England Premier League
3,West Ham United vs Wigan Athletic,England,England Premier League
4,Aston Villa vs Manchester City,England,England Premier League
...,...,...,...
8523,Atlético Madrid vs Valencia CF,Spain,Spain LIGA BBVA
8524,Málaga CF vs RC Deportivo de La Coruña,Spain,Spain LIGA BBVA
8525,Athletic Club de Bilbao vs Real Sporting de Gijón,Spain,Spain LIGA BBVA
8526,Granada CF vs Real Betis Balompié,Spain,Spain LIGA BBVA


**23. Match Classifications** 
"Create a report that lists match details and adds a column called `Result`. It should say 'Home Win', 'Away Win', or 'Draw' based on the scores."

**Expected Output:**
| match | home_goal | away_goal | Result |
|---|---|---|---|
| Liverbool vs Man City  | 1 | 1 | Draw |
| Liverbool vs Man United  | 2 | 0 | Home Win |
| ... | ... | ... | ... |

In [142]:
pd.read_sql(
  """
    SELECT 
      CONCAT(HT.team_long_name, ' vs ', AT.team_long_name) AS match,
      M.home_team_goal AS home_goal,
      M.away_team_goal AS away_goal,
      CASE 
        WHEN (M.home_team_goal = M.away_team_goal) THEN 'Draw'
        WHEN (M.home_team_goal > M.away_team_goal) THEN 'Home Win'
        ELSE 'Away Win'
      END AS Result
    FROM Match AS M
    JOIN Team AS HT
      ON M.home_team_api_id = HT.team_api_id
    JOIN Team AS AT
      ON M.away_team_api_id = AT.team_api_id
  """,
  conn
)

,match,home_goal,away_goal,Result
0,KRC Genk vs Beerschot AC,1,1,Draw
1,SV Zulte-Waregem vs Sporting Lokeren,0,0,Draw
2,KSV Cercle Brugge vs RSC Anderlecht,0,3,Away Win
3,KAA Gent vs RAEC Mons,5,0,Home Win
4,FCV Dender EH vs Standard de Liège,1,3,Away Win
...,...,...,...,...
25974,FC St. Gallen vs FC Thun,1,0,Home Win
25975,FC Vaduz vs FC Luzern,1,2,Away Win
25976,Grasshopper Club Zürich vs FC Sion,2,0,Home Win
25977,Lugano vs FC Zürich,0,0,Draw


**24. The December Boys** 
"We are investigating the 'Relative Age Effect'. Find players born in **December**. You'll need to extract the month from the `birthday` column."

**Expected Output:**
| player_name | birthday |
|---|---|
| Aaron Ramsey | 1990-12-26 |
| ... | ... |

In [172]:
pd.read_sql(
  """
    SELECT 
      player_name, 
      DATE(birthday) AS birthday
    FROM Player
    WHERE STRFTIME('%m', birthday) = '12'
  """,
  conn
)

,player_name,birthday
0,Aaron Cresswell,1989-12-15
1,Aaron Ramsey,1990-12-26
2,Abdelkader Ghezzal,1984-12-05
3,Abdellah Zoubir,1991-12-05
4,"Abdoulaye Diallo Sadio,22",1990-12-28
...,...,...
687,Zdenek Ondrasek,1988-12-22
688,Zdenek Pospech,1978-12-14
689,Zeljko Kalac,1972-12-16
690,Zlatan Ljubijankic,1983-12-15


**25. High Impact Games** 
"List the **League Names** and the **Match Dates** where the total score (Home + Away) was 7 or higher."

**Expected Output:**
| League_Name | date | Total_Score |
|---|---|---|
| Premier League | 2012-08-17 | 8 |
| ... | ... | ... |

In [174]:
pd.read_sql(
  """
    SELECT
      L.name AS League_Name,
      DATE(M.date) AS Date,
      (M.home_team_goal + M.away_team_goal) AS Total_Score
    FROM Match AS M
    JOIN League AS L
      ON M.league_id = L.id
    WHERE (M.home_team_goal + M.away_team_goal) >= 7
  """,
  conn
)

,League_Name,Date,Total_Score
0,Belgium Jupiler League,2008-12-06,7
1,Belgium Jupiler League,2008-12-20,7
2,Belgium Jupiler League,2009-02-01,7
3,Belgium Jupiler League,2009-03-07,7
4,Belgium Jupiler League,2008-10-25,8
...,...,...,...
547,Switzerland Super League,2016-04-17,7
548,Switzerland Super League,2016-04-17,9
549,Switzerland Super League,2015-08-08,7
550,Switzerland Super League,2015-08-22,7


**26. The Ratings Sheet** 
"List each player's **Name** alongside their **max overall_rating** recorded in the database."

**Expected Output:**
| player_name | Max_Rating |
|---|---|
| Lionel Messi | 94 |
| Cristiano Ronaldo | 93 |
| ... | ... |

In [7]:
pd.read_sql(
  """
    SELECT 
      player_name,
      MAX(overall_rating) AS Max_Rating
    FROM Player AS P
    JOIN Player_Attributes AS PA
      ON P.player_api_id = PA.player_api_id
    GROUP BY player_name
    ORDER BY Max_Rating DESC
  """,
  conn
)

,player_name,Max_Rating
0,Lionel Messi,94
1,Wayne Rooney,93
2,Gianluigi Buffon,93
3,Cristiano Ronaldo,93
4,Xavi Hernandez,92
...,...,...
10843,Jordan Kirkpatrick,47
10844,Gianluca D'Angelo,47
10845,Liam Hughes,46
10846,Benjamin Fischer,46


**27. Tactical Analysis** 
"We are studying 'Manchester United'. Join the `Team` and `Team_Attributes` tables to show their `buildUpPlaySpeed` scores over time."

**Expected Output:**
| team_long_name | date | buildUpPlaySpeed |
|---|---|---|
| Manchester United | 2010-02-22 | 70 |
| Manchester United | 2011-02-22 | 65 |
| ... | ... | ... |

In [19]:
pd.read_sql(
  """
    SELECT 
      T.team_long_name,
      DATE(date) AS Date,
      TA.buildUpPlaySpeed
    FROM Team AS T
    JOIN Team_Attributes AS TA
      ON T.team_api_id = TA.team_api_id
    WHERE T.team_long_name = 'Manchester United'
  """,
  conn
)

,team_long_name,Date,buildUpPlaySpeed
0,Manchester United,2010-02-22,70
1,Manchester United,2011-02-22,65
2,Manchester United,2012-02-22,46
3,Manchester United,2013-09-20,46
4,Manchester United,2014-09-19,46
5,Manchester United,2015-09-10,38


**28. The Blowouts** 
"For every match, calculate the `Goal_Difference` (absolute difference between Home and Away goals). Sort the list to show the biggest blowouts first."

**Expected Output:**
| match | Goal_Difference |
|---|---|
| Liverbool vs Man City | 9 |
| Liverbool vs Man United | 8 |
| ... | ... |

In [22]:
pd.read_sql(
  """
    SELECT 
      CONCAT(HT.team_long_name, ' vs ', AT.team_long_name) AS match,
      ABS(home_team_goal - away_team_goal) AS Goal_Difference
    FROM Match AS M
    JOIN Team AS HT
      ON M.home_team_api_id = HT.team_api_id
    JOIN Team AS AT
      ON M.away_team_api_id = AT.team_api_id
    ORDER BY Goal_Difference DESC
  """,
  conn
)

,match,Goal_Difference
0,PSV vs Feyenoord,10
1,ES Troyes AC vs Paris Saint-Germain,9
2,Celtic vs Aberdeen,9
3,Tottenham Hotspur vs Wigan Athletic,8
4,Chelsea vs Wigan Athletic,8
...,...,...
25974,FC Vaduz vs FC Sion,0
25975,FC Vaduz vs Grasshopper Club Zürich,0
25976,FC Luzern vs Grasshopper Club Zürich,0
25977,FC Zürich vs FC Thun,0


**29. El Clásico History** 
"Find all matches played between 'FC Barcelona' (Home) and 'Real Madrid CF' (Away)."

**Expected Output:**
| date | Home_Team | Away_Team | home_goal | away_goal |
|---|---|---|---|---|
| 2010-11-29 | FC Barcelona | Real Madrid CF | 5 | 0 |
| ... | ... | ... | ... | ... |

In [30]:
pd.read_sql(
  """
    SELECT 
      DATE(date) AS Date,
      HT.team_long_name AS Home_Team, 
      AT.team_long_name AS Away_Team,
      M.home_team_goal AS home_goal,
      M.away_team_goal AS away_goal
    FROM Match AS M
    JOIN Team AS HT
      ON M.home_team_api_id = HT.team_api_id
    JOIN Team AS AT
      ON M.away_team_api_id = AT.team_api_id
    WHERE (Home_Team = 'FC Barcelona' AND Away_Team = 'Real Madrid CF')
  """,
  conn
)

,Date,Home_Team,Away_Team,home_goal,away_goal
0,2008-12-13,FC Barcelona,Real Madrid CF,2,0
1,2009-11-29,FC Barcelona,Real Madrid CF,1,0
2,2010-11-29,FC Barcelona,Real Madrid CF,5,0
3,2012-04-21,FC Barcelona,Real Madrid CF,1,2
4,2012-10-07,FC Barcelona,Real Madrid CF,2,2
5,2013-10-26,FC Barcelona,Real Madrid CF,2,1
6,2015-03-22,FC Barcelona,Real Madrid CF,2,1
7,2016-04-02,FC Barcelona,Real Madrid CF,1,2


**30. The Snooze Fests** 
"Which League has the most boring games? Count how many matches ended **0-0** per League Name."

**Expected Output:**
| League_Name | Boring_Matches |Match|
|---|---|---|
| France Ligue 1 | 299 |PSG vs Lile|
| England Premier League | 200 |Liverbool vs Man City|
| ... | ... | ... |

In [34]:
pd.read_sql(
  """
    SELECT 
      L.name AS League_Name,
      COUNT(*) AS Boring_Matches,
      CONCAT(HT.team_long_name, ' vs ', AT.team_long_name) AS Match
    FROM Match AS M
    JOIN League AS L
      ON M.league_id = L.id
    JOIN Team AS HT
      ON M.home_team_api_id = HT.team_api_id
    JOIN Team AS AT
      ON M.away_team_api_id = AT.team_api_id
    WHERE (M.home_team_goal = 0) AND (M.away_team_goal = 0)
    GROUP BY League_Name
  """,
  conn
)

,League_Name,Boring_Matches,Match
0,Belgium Jupiler League,115,SV Zulte-Waregem vs Sporting Lokeren
1,England Premier League,251,Wigan Athletic vs Stoke City
2,France Ligue 1,277,AS Nancy-Lorraine vs LOSC Lille
3,Germany 1. Bundesliga,164,VfL Bochum vs SV Werder Bremen
4,Italy Serie A,240,Atalanta vs Lecce
5,Netherlands Eredivisie,119,Heracles Almelo vs Vitesse
6,Poland Ekstraklasa,178,Polonia Bytom vs Ruch Chorzów
7,Portugal Liga ZON Sagres,185,Trofense vs SC Braga
8,Scotland Premier League,141,St. Mirren vs Hibernian
9,Spain LIGA BBVA,221,CA Osasuna vs Atlético Madrid


--- 
## 📋 Section 4: The Coaching Staff
*Advanced requests involving comparisons, rankings, and deep analysis.*

**31. The Tallest of the Tall** 
"Select the names of players whose height is **greater than the average height** of all players in the database."

**Expected Output:**
| player_name | height | Team|
|---|---|---|
| Peter Crouch | 201 |Alahly|
| Zlatan Ibrahimovic | 195 |Milan|
| ... | ... | ... |

In [48]:
pd.read_sql(
  """
    SELECT 
      player_name,
      height
    FROM Player
    WHERE height > (SELECT AVG(height) FROM Player)
  """,
  conn
)

,player_name,height
0,Aaron Appindangoye,182.88
1,Aaron Galindo,182.88
2,Aaron Hughes,182.88
3,Aaron Hunt,182.88
4,Aaron Lennox,190.50
...,...,...
5865,Zoran Rendulic,190.50
5866,Zouhair Feddal,190.50
5867,Zoumana Camara,182.88
5868,Zsolt Laczko,182.88


**32. The Points Table** 
"Calculate points for every match in 2014/2015 (3 for win, 1 for draw, 0 for loss). Group by team and find the total points for the season."

**Expected Output:**
| Team_Name | Total_Points |
|---|---|
| Chelsea | 87 |
| Manchester City | 79 |
| ... | ... |

In [73]:
pd.read_sql(
  """
    WITH All_Matches AS (
      SELECT 
        home_team_api_id AS team, 
        home_team_goal AS goals_for, 
        away_team_goal AS goals_against, 
        season
      FROM Match
      UNION ALL
      SELECT 
        away_team_api_id AS team, 
        away_team_goal AS goals_for, 
        home_team_goal AS goals_against, 
        season
      FROM Match
    )
    SELECT
      T.team_long_name AS Team_Name,
      SUM(
        CASE
          WHEN goals_for > goals_against THEN 3
          WHEN goals_for = goals_against THEN 1
          ELSE 0
        END
      ) AS Total_Points
    FROM Team AS T
    JOIN All_Matches AS A
      ON T.team_api_id = A.team
    WHERE season = '2014/2015'
    GROUP BY Team_Name
  """,
  conn
)

,Team_Name,Total_Points
0,1. FC Köln,40
1,1. FSV Mainz 05,40
2,ADO Den Haag,37
3,AS Monaco,71
4,AS Saint-Étienne,69
...,...,...
183,Willem II,46
184,Wisła Kraków,43
185,Zawisza Bydgoszcz,29
186,Évian Thonon Gaillard FC,37


**33. Momentum Builder** 
"We need to analyze team form. Calculate the **running total of goals** scored by 'Arsenal' (as home team) throughout the 2015/2016 season, ordered by date."

**Expected Output:**
| date | home_goal | Running_Total |
|---|---|---|
| 2015-08-09 | 0 | 0 |
| 2015-08-24 | 0 | 0 |
| 2015-09-12 | 2 | 2 |
| ... | ... | ... |

In [80]:
pd.read_sql(
  """
    SELECT 
      DATE(date) AS Date,
      M.home_team_goal AS Home_Goal,
      SUM(M.home_team_goal) OVER(ORDER BY Date ASC) AS Running_Total
    FROM Match AS M
    JOIN Team AS T
      ON M.home_team_api_id = T.team_api_id
    WHERE T.team_long_name = 'Arsenal' AND season = '2015/2016'
  """,
  conn
)

,Date,Home_Goal,Running_Total
0,2015-08-09,0,0
1,2015-08-24,0,0
2,2015-09-12,2,2
3,2015-10-04,3,5
4,2015-10-24,2,7
5,2015-11-08,1,8
6,2015-12-05,3,11
7,2015-12-21,2,13
8,2015-12-28,2,15
9,2016-01-02,1,16


**34. The Offensive Power Rankings** 
"Rank all teams in the 2015/2016 season based on their **Total Goals Scored**. Use a ranking function."

**Expected Output:**
| Team_Name | Total_Goals | Rank |
|---|---|---|
| Real Madrid CF | 110 | 1 |
| FC Barcelona | 108 | 2 |
| ... | ... | ... |

In [134]:
pd.read_sql(
  """
    WITH All_Matches AS (
      SELECT 
        home_team_api_id AS team, 
        home_team_goal AS goals_for, 
        season
      FROM Match
      UNION ALL
      SELECT 
        away_team_api_id AS team, 
        away_team_goal AS goals_for, 
        season
      FROM Match
    )
    SELECT
      T.team_long_name AS Team_Name,
      SUM(goals_for) AS Total_Goals,
      RANK() OVER(ORDER BY SUM(goals_for) DESC) AS Rank
    FROM Team AS T
    JOIN All_Matches AS A
      ON T.team_api_id = A.team
    WHERE season = '2015/2016'
    GROUP BY Team_Name
  """,
  conn
)

,Team_Name,Total_Goals,Rank
0,FC Barcelona,112,1
1,Real Madrid CF,110,2
2,Paris Saint-Germain,102,3
3,Celtic,93,4
4,SL Benfica,88,5
...,...,...,...
183,Sint-Truidense VV,28,183
184,ES Troyes AC,28,183
185,Uniao da Madeira,27,186
186,Aston Villa,27,186


**35. The Most Entertaining League** 
"Which league has the highest average goals per match? Aggregate the totals first, then find the max."

**Expected Output:**
| League_Name | Avg_Goals_Per_Match |Match|
|---|---|---|
| Switzerland Super League | 3.14 |PCV vs Halona|
| Netherlands Eredivisie | 3.08 |Grec vs Maro|
| ... | ... | ...|

In [104]:
pd.read_sql(
  """
    -- ((أسماء الماتشات هنا مش مطلوب صحيح في ال Expected Output))
    SELECT
      name AS League_Name,
      ROUND(AVG(M.home_team_goal + M.away_team_goal), 2) AS Avg_Goals_Per_Match,
      CONCAT(HT.team_long_name, ' vs ', AT.team_long_name) AS Match
    FROM Match AS M
    JOIN League AS L
      ON M.league_id = L.id
    JOIN Team AS HT
      ON M.home_team_api_id = HT.team_api_id
    JOIN Team AS AT
      ON M.away_team_api_id = AT.team_api_id
    GROUP BY League_Name
    ORDER BY Avg_Goals_Per_Match DESC
    LIMIT 1
  """,
  conn
)

,League_Name,Avg_Goals_Per_Match,Match
0,Netherlands Eredivisie,3.08,Vitesse vs FC Groningen


**36. The Fortress** 
"Calculate the **percentage of wins** for the Home Team for each specific league."

**Expected Output:**
| League_Name | Home_Win_Pct | Home_Team|
|---|---|---|
| Spain LIGA BBVA | 49.2 | MAdrid|
| England Premier League | 46.5 |Man City|
| ... | ... | ... |

In [148]:
pd.read_sql(
  """
    SELECT
      L.name AS League_Name,
      ROUND((SUM(CASE WHEN M.home_team_goal > M.away_team_goal THEN 1 ELSE 0 END))*100.0/
      (COUNT(*)), 2) AS Home_Win_Pct,
      T.team_long_name AS Home_Team
    FROM Match AS M
    JOIN League AS L
      ON M.league_id = L.id
    JOIN Team AS T
      ON T.team_api_id = M.home_team_api_id
    GROUP BY Home_Team
  """,
  conn
)

,League_Name,Home_Win_Pct,Home_Team
0,Germany 1. Bundesliga,23.53,1. FC Kaiserslautern
1,Germany 1. Bundesliga,31.37,1. FC Köln
2,Germany 1. Bundesliga,35.29,1. FC Nürnberg
3,Germany 1. Bundesliga,46.22,1. FSV Mainz 05
4,France Ligue 1,28.07,AC Ajaccio
...,...,...,...
291,Spain LIGA BBVA,31.58,Xerez Club Deportivo
292,Poland Ekstraklasa,38.89,Zagłębie Lubin
293,Poland Ekstraklasa,43.33,Zawisza Bydgoszcz
294,France Ligue 1,38.16,Évian Thonon Gaillard FC


**37. Most Improved** 
"Look at the `Player_Attributes`. Find players who increased their `overall_rating` by **5 or more points** in a single update compared to their previous record."

**Expected Output:**
| player_name | Old_Rating | New_Rating | Jump |
|---|---|---|---|
| Jamie Vardy | 72 | 78 | 6 |
| Riyad Mahrez | 74 | 80 | 6 |
| ... | ... | ... | ... |

In [197]:
pd.read_sql(
  """
    WITH T AS(
      SELECT 
      P.player_name AS Player_Name,
      PA.overall_rating AS New_Rating,
      LAG(PA.overall_rating) OVER(PARTITION BY P.player_api_id ORDER BY date) AS Old_Rating
      FROM Player AS P
      JOIN Player_Attributes AS PA
        ON P.player_api_id = PA.player_api_id
    )
    SELECT *, (New_Rating - Old_Rating) AS Jump
    FROM T
    WHERE (New_Rating - Old_Rating) >= 5
  """,
  conn
)

,Player_Name,New_Rating,Old_Rating,Jump
0,Jose Dorado,65,58,7
1,Jose Dorado,70,65,5
2,Pablo Hernandez,78,67,11
3,Ruben Perez,72,60,12
4,Jorge Molina,73,68,5
...,...,...,...,...
5370,Ousmane Dembele,62,57,5
5371,Cameron Borthwick-Jackson,67,62,5
5372,Marcus Rashford,65,59,6
5373,Issa Diop,66,60,6


**38. The League Leaders** 
"Find the Top 3 scoring teams (Total Home Goals) for **each** league in a single query."

**Expected Output:**
| League_Name | Team_Name | Goals | Rank |
|---|---|---|---|
| England | Man City | 54 | 1 |
| England | Arsenal | 48 | 2 |
| England | Chelsea | 47 | 3 |
| Spain | Real Madrid | 60 | 1 |
| ... | ... | ... | ... |

In [2]:
pd.read_sql(
  """
    WITH T1 AS (
      SELECT 
        L.name AS League_Name,
        T.team_long_name AS Team_Name,
        SUM(M.home_team_goal) AS Goals
      FROM Team AS T
      JOIN Match AS M
        ON T.team_api_id = M.home_team_api_id
      JOIN League AS L
        ON M.league_id = L.id
      GROUP BY T.team_long_name
      ORDER BY Goals DESC
    ),
    T2 AS (
      SELECT *, ROW_NUMBER() OVER(PARTITION BY League_Name ORDER BY Goals DESC) AS Rank
      FROM T1
    )
    SELECT *
    FROM T2
    WHERE Rank <= 3
    ORDER BY League_Name
  """,
  conn
)

,League_Name,Team_Name,Goals,Rank
0,Belgium Jupiler League,RSC Anderlecht,247,1
1,Belgium Jupiler League,Club Brugge KV,235,2
2,Belgium Jupiler League,KAA Gent,213,3
3,England Premier League,Manchester City,365,1
4,England Premier League,Manchester United,338,2
5,England Premier League,Chelsea,333,3
6,France Ligue 1,Paris Saint-Germain,332,1
7,France Ligue 1,Olympique Lyonnais,289,2
8,France Ligue 1,LOSC Lille,270,3
9,Germany 1. Bundesliga,FC Bayern Munich,382,1


**39. Consistency Kings** 
"Find teams that scored **at least 1 goal** in every single home match they played in the 2015/2016 season."

**Expected Output:**
| Team_Name |Match|Goals|
|---|---|---|
| Real Madrid CF |...vs...|5
| Borussia Dortmund |...vs...|3
| ... |...|...|

In [2]:
pd.read_sql(
  """
    SELECT 
      T.team_long_name AS Team_Name,
      SUM(M.home_team_goal) AS Goals
    FROM Match AS M
    JOIN Team AS T
      ON M.home_team_api_id = T.team_api_id
    WHERE season = '2015/2016'
    GROUP BY Team_Name
    HAVING MIN(M.home_team_goal) >= 1
  """,
  conn
)

,Team_Name,Goals
0,Club Brugge KV,42
1,FC Barcelona,67
2,FC Bayern Munich,51
3,KAA Gent,35


**40. The Relegated/Promoted** 
"Find teams that appear in the Match table for the '2008/2009' season but do **NOT** appear in the '2015/2016' season."

**Expected Output:**
| Team_Name |
|---|
| Portsmouth |
| Hull City |
| ... |

In [11]:
pd.read_sql(
  """
    SELECT DISTINCT T.team_long_name AS Team_Name
    FROM Match AS M
    JOIN Team AS T
      ON M.home_team_api_id = T.team_api_id
    WHERE season = '2008/2009' AND M.home_team_api_id NOT IN (SELECT M2.home_team_api_id FROM Match AS M2 WHERE season = '2015/2016')
  """,
  conn
)

,Team_Name
0,KSV Cercle Brugge
1,FCV Dender EH
2,KSV Roeselare
3,Tubize
4,Royal Excel Mouscron
5,Beerschot AC
6,RAEC Mons
7,Middlesbrough
8,Bolton Wanderers
9,Hull City


--- 
## 🛠️ Section 5: The Technical Architect
*Building tools to make future analysis faster and cleaner.*

**41. The Readable Match View** 
"Create a `VIEW` called `v_Match_Details` that joins Match, Country, League, Home_Team, and Away_Team to display readable names instead of IDs."

**Expected Output:**
*(No output expected, just the creation command. Verify by selecting 5 rows from view)*

In [99]:
conn.execute(
  """
    CREATE VIEW v_Match_Details 
    AS
      SELECT 
        M.match_api_id AS Match_api_id,
        M.season AS Season,
        HT.team_long_name AS Home_Team,
        AT.team_long_name AS Away_Team,
        L.name AS League_Name,
        C.name AS Country_Name
      FROM Match AS M
      JOIN Country AS C
        ON M.country_id = C.id
      JOIN League AS L
        ON M.league_id = L.id
      JOIN Team AS HT
        ON M.home_team_api_id = HT.team_api_id
      JOIN Team AS AT
        ON M.away_team_api_id = AT.team_api_id
  """
)

In [100]:
pd.read_sql(
  """
    SELECT * FROM v_Match_Details
  """,
  conn
)

,Match_api_id,Season,Home_Team,Away_Team,League_Name,Country_Name
0,492473,2008/2009,KRC Genk,Beerschot AC,Belgium Jupiler League,Belgium
1,492474,2008/2009,SV Zulte-Waregem,Sporting Lokeren,Belgium Jupiler League,Belgium
2,492475,2008/2009,KSV Cercle Brugge,RSC Anderlecht,Belgium Jupiler League,Belgium
3,492476,2008/2009,KAA Gent,RAEC Mons,Belgium Jupiler League,Belgium
4,492477,2008/2009,FCV Dender EH,Standard de Liège,Belgium Jupiler League,Belgium
...,...,...,...,...,...,...
25974,1992091,2015/2016,FC St. Gallen,FC Thun,Switzerland Super League,Switzerland
25975,1992092,2015/2016,FC Vaduz,FC Luzern,Switzerland Super League,Switzerland
25976,1992093,2015/2016,Grasshopper Club Zürich,FC Sion,Switzerland Super League,Switzerland
25977,1992094,2015/2016,Lugano,FC Zürich,Switzerland Super League,Switzerland


**42. The Latest Stats View** 
"Create a `VIEW` called `v_Player_Stats` showing only the **latest** available attributes for every player (filtering out old records)."

**Expected Output:**
*(No output expected, just the creation command)*

In [17]:
conn.execute(
  """
    CREATE VIEW v_Player_Stats
    AS
      SELECT *
        FROM (
        SELECT 
          *,
          ROW_NUMBER() OVER(PARTITION BY player_api_id ORDER BY date DESC) AS R
        FROM Player_Attributes
        ) AS T
      WHERE T.R = 1
  """
)

In [ ]:
pd.read_sql(
  """
    SELECT * FROM v_Player_Stats
  """,
  conn
)

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,R
0,139844,148544,2625,2015-01-16 00:00:00,61,61,right,medium,medium,50,...,66,62,63,54.0,12,11,6,8,8,1
1,44145,135819,2752,2015-10-16 00:00:00,72,72,right,medium,medium,39,...,38,72,74,67.0,12,7,8,10,16,1
2,88285,110019,2768,2016-03-17 00:00:00,74,74,left,medium,medium,44,...,16,76,76,77.0,12,15,13,14,10,1
3,72142,182861,2770,2013-07-05 00:00:00,69,69,right,medium,low,58,...,69,33,43,25.0,12,13,6,14,15,1
4,5122,110809,2790,2010-08-30 00:00:00,67,77,left,None,7,72,...,54,70,73,69.0,8,14,8,13,12,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11055,66602,226508,744907,2016-06-09 00:00:00,53,67,left,medium,medium,43,...,53,40,49,50.0,9,14,13,12,9,1
11056,58305,233930,746419,2016-05-12 00:00:00,59,66,right,high,medium,55,...,43,55,69,62.0,7,10,7,9,9,1
11057,60180,233969,748432,2016-05-12 00:00:00,58,68,right,medium,medium,48,...,45,63,69,68.0,8,8,12,12,6,1
11058,147409,225462,750435,2016-04-14 00:00:00,60,74,right,medium,low,35,...,61,18,19,21.0,9,10,8,10,11,1


**43. Team Match History (SP)** 
"Create a `STORED PROCEDURE` where I input a **Team Name**, and it returns all matches (home and away) played by that team."

**Expected Output:**
*(No output expected. Call the procedure with 'Arsenal' to test)*

In [101]:
def get_matches(team_name):
    return pd.read_sql(
        """
            SELECT *
            FROM v_Match_Details
            WHERE Home_Team = ? OR Away_Team = ?
        """, 
        conn, 
        params=[team_name, team_name]
    )

get_matches('KRC Genk')

,Match_api_id,Season,Home_Team,Away_Team,League_Name,Country_Name
0,492473,2008/2009,KRC Genk,Beerschot AC,Belgium Jupiler League,Belgium
1,492571,2008/2009,Tubize,KRC Genk,Belgium Jupiler League,Belgium
2,492580,2008/2009,KVC Westerlo,KRC Genk,Belgium Jupiler League,Belgium
3,492583,2008/2009,KRC Genk,KSV Roeselare,Belgium Jupiler League,Belgium
4,492628,2008/2009,KAA Gent,KRC Genk,Belgium Jupiler League,Belgium
...,...,...,...,...,...,...
207,1979871,2015/2016,Sint-Truidense VV,KRC Genk,Belgium Jupiler League,Belgium
208,1980066,2015/2016,KRC Genk,Sporting Charleroi,Belgium Jupiler League,Belgium
209,1979872,2015/2016,RSC Anderlecht,KRC Genk,Belgium Jupiler League,Belgium
210,1979882,2015/2016,KRC Genk,KV Mechelen,Belgium Jupiler League,Belgium


**44. Standings Generator (SP)** 
"Create a `STORED PROCEDURE` that takes a **Season** and **League_ID** and returns the table standings (Points, W, D, L) for that specific timeframe."

**Expected Output:**
*(No output expected. Call with '2015/2016' and '1729' to test)*

In [55]:
def get_team_standings(Season, League_ID):
  return pd.read_sql(
    """
      WITH All_Teams AS (
        SELECT home_team_api_id AS team, home_team_goal AS goals_for, away_team_goal AS goals_against
        FROM Match
        WHERE season = ? AND league_id = ?
        UNION ALL
        SELECT away_team_api_id AS team, away_team_goal AS goals_for, home_team_goal AS goals_against
        FROM Match
        WHERE season = ? AND league_id = ?
      ),
      T1 AS (
        SELECT team, goals_for, goals_against
        FROM All_Teams
      )
      SELECT 
        team,
        SUM(CASE WHEN goals_for > goals_against THEN 1 ELSE 0 END) AS W,
        SUM(CASE WHEN goals_for = goals_against THEN 1 ELSE 0 END) AS D,
        SUM(CASE WHEN goals_for < goals_against THEN 1 ELSE 0 END) AS L,
        SUM(
          CASE
            WHEN goals_for > goals_against THEN 3
            WHEN goals_for = goals_against THEN 1
            ELSE 0
          END
        ) AS Total_Points
      FROM T1
      GROUP BY team
    """,
    conn,
    params=[Season, League_ID, Season, League_ID]
  )

get_team_standings('2015/2016', '1729')

,team,W,D,L,Total_Points
0,8197,23,12,3,81
1,8455,12,14,12,50
2,8456,19,9,10,66
3,8466,18,9,11,63
4,8472,9,12,17,39
5,8586,19,13,6,70
6,8650,16,12,10,60
7,8654,16,14,8,62
8,8659,10,13,15,43
9,8668,11,14,13,47


**45. Scout Report (SP)** 
"Create a `STORED PROCEDURE` that accepts a **Min_Potential** value and returns players whose potential rating is above that threshold."

**Expected Output:**
*(No output expected. Call with '85' to test)*

In [152]:
def get_players(Min_Potential):
  return pd.read_sql(
    """
      SELECT *
      FROM Player_Attributes
      WHERE potential > ? 
    """,
    conn,
    params=[Min_Potential]
  )

get_players(90)

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,3250,106019,30690,2007-02-22 00:00:00,89,91,left,medium,low,53,...,78.0,86,12,18,20.0,9,8,49,9,8
1,4922,5879,30881,2007-02-22 00:00:00,85,91,right,medium,low,64,...,78.0,71,19,31,22.0,14,11,37,14,13
2,5991,216349,395154,2015-10-16 00:00:00,77,91,left,high,medium,68,...,81.0,75,41,53,55.0,15,11,11,13,10
3,5992,216349,395154,2015-10-02 00:00:00,77,91,left,high,medium,68,...,81.0,75,41,53,55.0,15,11,11,13,10
4,6220,1075,30727,2007-02-22 00:00:00,87,91,right,medium,low,92,...,90.0,90,26,45,32.0,11,16,82,16,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,183689,41236,35724,2009-02-22 00:00:00,89,94,right,medium,low,68,...,80.0,91,24,69,28.0,12,24,65,24,24
396,183690,41236,35724,2008-08-30 00:00:00,89,94,right,medium,low,68,...,80.0,91,24,69,28.0,12,24,65,24,24
397,183691,41236,35724,2008-02-22 00:00:00,87,91,right,medium,low,57,...,80.0,72,24,21,28.0,12,24,36,24,24
398,183692,41236,35724,2007-08-30 00:00:00,87,91,right,medium,low,57,...,80.0,72,24,21,28.0,12,24,36,24,24


**46. The Phantom Filter (Troubleshooting)** 
"We tried to filter groups by 'Average Goals > 2.5' using a WHERE clause and it failed. Fix the query using the correct keyword."

**Expected Output:**
| League_Name | Avg_Goals |
|---|---|
| Eredivisie | 3.1 |
| ... | ... |

In [61]:
pd.read_sql(
  """
    SELECT 
      L.name AS League_Name,
      ROUND(AVG(home_team_goal+away_team_goal), 2) AS Avg_Goals
    FROM Match AS M
    JOIN League AS L
      ON M.league_id = L.id
    GROUP BY L.name
    HAVING Avg_Goals > 2.5
  """,
  conn
)

,League_Name,Avg_Goals
0,Belgium Jupiler League,2.80
1,England Premier League,2.71
2,Germany 1. Bundesliga,2.90
3,Italy Serie A,2.62
4,Netherlands Eredivisie,3.08
5,Portugal Liga ZON Sagres,2.53
6,Scotland Premier League,2.63
7,Spain LIGA BBVA,2.77
8,Switzerland Super League,2.93


**47. Distinct Windows** 
"Calculate the number of **distinct opponents** a team has faced. *Warning: Do not try to use DISTINCT inside a Window Function, it will fail.*"

**Expected Output:**
| Team_Name | Distinct_Opponents |
|---|---|
| Arsenal | 45 |
| ... | ... |

In [71]:
pd.read_sql(
  """
    WITH all_matches AS (
      SELECT home_team_api_id AS team, away_team_api_id opponent
      FROM Match
      UNION ALL
      SELECT away_team_api_id AS team, home_team_api_id opponent
      FROM Match
    )
    SELECT T.team_long_name AS Team_Name, COUNT(DISTINCT opponent) AS Distinct_Opponents
    FROM all_matches M
    JOIN Team AS T
      ON M.team = T.team_api_id
    GROUP BY team
  """,
  conn
)

,Team_Name,Distinct_Opponents
0,Ruch Chorzów,23
1,Oud-Heverlee Leuven,19
2,Jagiellonia Białystok,23
3,S.C. Olhanense,22
4,Lech Poznań,23
...,...,...
294,FC Arouca,20
295,Termalica Bruk-Bet Nieciecza,15
296,Tondela,17
297,Carpi,19


**48. Handling Ghosts (NULLs)** 
"Calculate the average rating of players but explicitly treat NULLs as 0 using `COALESCE`, then compare it to a standard AVG."

**Expected Output:**
| Standard_Avg | Coalesce_Avg |
|---|---|
| 68.5 | 68.2 |
*(Notice the difference?)*

In [89]:
pd.read_sql(
  """
    SELECT 
      ROUND(AVG(overall_rating), 1) AS Standard_Avg,
      ROUND(AVG(COALESCE(overall_rating, 0)), 1) AS Coalesce_Avg
    FROM Player_Attributes
  """,
  conn
)

,Standard_Avg,Coalesce_Avg
0,68.6,68.3


**49. The Cartesian Nightmare** 
"Demonstrate a 'Cross Join' between Team and League (every team in every league). Count the rows to show why we must always link tables correctly."

**Expected Output:**
| Cartesian_Count |
|---|
| 25000+ |
*(Should be huge compared to real data)*

In [83]:
pd.read_sql(
  """
    SELECT COUNT(*) AS [Cartesian_Count (299 * 11)]
    FROM Team , League
  """,
  conn
)

,Cartesian_Count (299 * 11)
0,3289


**50. The Final Exam** 
"Write a single query using SELECT, FROM, WHERE, GROUP BY, HAVING, and ORDER BY. Comment above each line the order (1-6) in which the engine processes them."

**Expected Output:**
*(Correct query execution with no errors)*

In [ ]:
pd.read_sql(
  """
    -- 5
    SELECT league_id, SUM(home_team_goal + away_team_goal) AS Total_Goals
    -- 1
    FROM Match
    -- 2
    WHERE season = '2008/2009'
    -- 3
    GROUP BY league_id
    -- 4
    HAVING Total_Goals > 50
    -- 6
    ORDER BY Total_Goals
  """,
  conn
)

,league_id,Total_Goals
0,15722,524
1,24558,540
2,19694,548
3,17642,552
4,1,855
5,4769,858
6,13274,870
7,7809,894
8,1729,942
9,10257,988
